# NBA 3.1 - Write HCP-level input files (Dx_09, Dx_03, Rx_01_new)

This notebook pulls the inputs for the HCP-level feature notebooks from Snowflake (IQVIA LAAD) into the **NBA 3.1 volume**. It keeps the NBA 3.0 SQL from `00_Write_Input_Files` and changes only the HCP universe, the date range and the paths.

| File | Used by | Source | Written to |
| --- | --- | --- | --- |
| `Dx_09` | `09_claim_comorbidity` | `STG_CAR_IQV_LAAD_FACT_DX` x `PHCDW_SANDBOX.FIN_DIAG_CD` (KD, DB-KD, CV-KD) | `kerendia_nba_3_1_volume/hcp_level_model/ops_pipeline/query_output/Dx_09` |
| `Dx_03` | `11_adoption_features_inv_far` | `STG_CAR_IQV_LAAD_FACT_DX` x `STG_IQVIA_LAAD_DIM_DIAGNOSIS` (CKD code list) | `.../query_output/Dx_03` |
| `Rx_01_new` | `11_adoption_features_inv_far` | `STG_CAR_IQV_LAAD_FCT_RX` x `PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW` | `.../query_output/Rx_01_new` |

**Driven by**
* Cohorts: `kerendia_nba_3_1_volume/cohorts/claims/cohorts.parquet` - the last cohort's `lookback_end_date` sets the last month pulled.
* HCP universe: `kerendia_nba_3_1_volume/HCP_Universe/` - every file is inner-joined to it.

**Differences from 3.0**
* 3.1 universe instead of the 3.0 universe.
* `END_MONTH` comes from the cohorts instead of the control dates. It is also applied to `Rx_01_new`, which had no upper date bound in 3.0. This keeps the static adoption features within the cohort window.
* `ORDER BY` removed from the Rx query because it only adds run time.

**Run order:** cells top to bottom. Run the Snowflake access check first. `Rx_01_new` is the largest pull (about 4B rows in 3.0); set `RX_INV_FAR_ONLY = True` if only notebook 11 needs it.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
import json
import pandas as pd
import pyspark.sql.functions as F

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
COHORTS_DIR = f"{VOL31}/cohorts/claims"
HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe"
QUERY_OUTPUT_31 = f"{VOL31}/hcp_level_model/ops_pipeline/query_output"

# Full history start (3.0 config value) - running totals in 09 start here
ANALYSIS_START_MONTH = 201901

# Optional: set e.g. 202609 to pull beyond the last cohort (inference). None = use cohorts.
END_MONTH_OVERRIDE = None

# Which files to pull (job parameter `files`, comma-separated; default = all three)
dbutils.widgets.text("files", "Dx_09,Dx_03,Rx_01_new")
FILES = [f.strip() for f in dbutils.widgets.get("files").split(",") if f.strip()]
PULL_DX_09 = "Dx_09" in FILES
PULL_DX_03 = "Dx_03" in FILES
PULL_RX_01_NEW = "Rx_01_new" in FILES

# True = keep only the adoption-feature products in Rx_01_new (notebook 11, much smaller file)
# Drug groups (NBA_3_1_Adoption_Features_Inv_Far_Recommendations.md, 1.0): Invokana, Farxiga, Jardiance,
# steroidal MRA (spironolactone + eplerenone), Entresto
RX_ADOPTION_ONLY = True
ADOPTION_PRODUCTS = ["INVOKANA", "FARXIGA", "JARDIANCE", "SPIRONOLACTONE", "EPLERENONE", "ENTRESTO"]
print(f"Files to pull: {FILES} | Rx_01_new adoption products only: {RX_ADOPTION_ONLY}")

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)

assert meta["feature_family"] == "claims", meta["feature_family"]
assert len(cohorts) == meta["cohort_count"], (len(cohorts), meta["cohort_count"])
assert (cohorts["offset_days"] == 0).all()

lookback_end_max = pd.to_datetime(cohorts["lookback_end_date"]).max()
lookback_start_min = pd.to_datetime(cohorts["lookback_start_date"]).min()
END_MONTH = END_MONTH_OVERRIDE or int(lookback_end_max.strftime("%Y%m"))
END_DATE = (pd.Period(str(END_MONTH), freq="M").to_timestamp(how="end")).date()   # last day of END_MONTH

print(f"Cohorts: {len(cohorts)} ({cohorts.cohort_id.iloc[0]} .. {cohorts.cohort_id.iloc[-1]}), generated {meta['generation_timestamp']}")
print(f"Earliest lookback start: {lookback_start_min.date()} | latest lookback end: {lookback_end_max.date()}")
print(f"Pull window: {ANALYSIS_START_MONTH} .. {END_MONTH} (last day {END_DATE})")
display(cohorts)

In [0]:
hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
n_hcp = hcp_universe.count()
assert dict(hcp_universe.dtypes)["PROVIDER_ID"] == "string"
assert hcp_universe.filter(F.col("PROVIDER_ID").isNull()).count() == 0
print(f"HCP universe: {n_hcp:,} distinct PROVIDER_IDs")

def save_input(df, name):
    """Inner-join to the 3.1 universe, add PULL_DT, overwrite Delta in the 3.1 query_output folder."""
    out = (df.join(F.broadcast(hcp_universe), on="PROVIDER_ID", how="inner")
             .withColumn("PULL_DT", F.current_date()))
    path = f"{QUERY_OUTPUT_31}/{name}"
    out.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    print(f"Saved {name} -> {path}")

In [0]:
sf_objects = [
    "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX",
    "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX",
    "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR",
    "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC",
    "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD",
    "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PLN",
    "PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS",
    "PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW",
    "PHCDW.PHCDW_SANDBOX.FIN_DIAG_CD",
]
access = {}
for obj in sf_objects:
    try:
        Get_Data_Snowflakes(f"SELECT 1 AS OK FROM {obj} LIMIT 1").collect()
        access[obj] = "OK"
    except Exception as e:
        access[obj] = "FAILED: " + str(e).splitlines()[0][:200]
display(pd.DataFrame(access.items(), columns=["object", "status"]))

# Latest data available in LAAD vs the pull window
latest = Get_Data_Snowflakes("""
    SELECT (SELECT MAX(SERVICE_DATE) FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX) AS MAX_DX_DATE,
           (SELECT MAX(SVC_DT) FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX) AS MAX_RX_DATE""").collect()[0]
print(f"LAAD max Dx date: {latest['MAX_DX_DATE']} | max Rx date: {latest['MAX_RX_DATE']} | pull ends: {END_DATE}")
assert all(v == "OK" for v in access.values()), "Fix Snowflake access before pulling"

### File Dx_09 (claim comorbidity)
Distinct patient x Dx date x description rows for `FIN_DIAG_CD` groups `KD`, `DB-KD`, `CV-KD`. `PROVIDER_ID = COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID)`.

In [0]:
if PULL_DX_09:
    query = f"""
        SELECT DISTINCT PATIENT_ID, SVC_DT, DIAGNOSIS_DESCRIPTION, DIAG_GRP, PROVIDER_ID, PROVIDER_RENDERING_ID, PROVIDER_REFERRING_ID FROM
        (
            SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS SVC_DT, CD.DIAGNOSIS_DESCRIPTION, DIAG_GRP, PROVIDER_RENDERING_ID, PROVIDER_REFERRING_ID, PROVIDER_ID FROM
            (
                SELECT PATIENT_ID, DIAGNOSIS_CODE, SERVICE_DATE, PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID,
                       COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID) AS PROVIDER_ID
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX
            ) DX
            INNER JOIN
            (
                SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION, DIAG_GRP
                FROM PHCDW.PHCDW_SANDBOX.FIN_DIAG_CD
                WHERE DIAG_GRP IN ('KD', 'CV-KD', 'DB-KD')
            ) CD
            ON DX.DIAGNOSIS_CODE = CD.DIAGNOSIS_CODE
            WHERE (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) >= {ANALYSIS_START_MONTH}
              AND (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) <= {END_MONTH}
              AND PROVIDER_ID IS NOT NULL
        )
    """
    save_input(Get_Data_Snowflakes(query), "Dx_09")

### File Dx_03 (CKD diagnoses for adoption features)
Verbatim 3.0 code list (163 codes). `DX_STAGE` 1-5 from the description, else 0.

In [0]:
if PULL_DX_03:
    DX03_CODES = """'249.4', '249.40', '249.41', '250.4', '250.40', '250.42', '285.21', '403', '403.0', '403.00', '403.1', '403.10', '403.9',
        '403.90', '404', '404.0', '404.00', '404.01', '404.1', '404.10', '404.11', '404.9', '404.90', '404.91', '585', '585.1',
        '585.2', '585.3', '585.4', '585.9', '588.81', 'D63.1', 'E08.2', 'E08.21','E08.22', 'E08.29', 'E09.2', 'E09.21', 'E09.22',
        'E09.29', 'E11.2', 'E11.21', 'E11.22', 'E11.29', 'E13.2', 'E13.21', 'E13.22', 'E13.29', 'I12', 'I12.9', 'I13', 'I13.0',
        'I13.1', 'I13.10', 'N18', 'N18.1', 'N18.2', 'N18.3', 'N18.4', 'N18.9', 'N25.81', '503.60' , '503.65', 'S20.65', '0TY0.0Z0',
        '0TY0.0Z1', '0TY0.0Z2', '0TY1.0Z0', '0TY1.0Z1', '0TY1.0Z2', 'Z94.0', 'V42.0', '556.9', '909.18', '909.19', '909.20', '909.21',
        '909.22', '909.23', '909.24', '909.25', '909.35', '909.37', '909.45', '909.47', '909.51', '909.52', '909.53', '909.54',
        '909.55', '909.56', '909.57', '909.58', '909.59', '909.60', '909.61', '909.62', '909.63', '909.64', '909.65', '909.66',
        '909.67', '909.68', '909.69', '909.70', '909.99', '995.12', '995.59', 'G02.57', 'G03.08', 'G03.09', 'G03.10', 'G03.11',
        'G03.12', 'G03.13', 'G03.14', 'G03.15', 'G03.16', 'G03.17', 'G03.18', 'G03.19', 'G03.20', 'G03.21', 'G03.22', 'G03.23',
        'G03.24', 'G03.25', 'G03.26', 'G03.27', 'G90.13', 'G90.14', 'S93.35', 'S93.39','3E1.M39Z', '5A1.D00Z', '5A1.D60Z',
        '5A1.D70Z', '5A1.D80Z', '5A1.D90Z', 'Z99.2', '399.5', '549.8', '800', '801', '802', '803', '804', '809', '820', '821',
        '829', '830', '831', '839', '840', '841', '849', '850', '851', '859', '880', '881', '889'"""

    query = f"""
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS SVC_DT, CD.DIAGNOSIS_DESCRIPTION AS Category,
            CASE
                WHEN CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 1' THEN 1
                WHEN CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 2 (MILD)' THEN 2
                WHEN CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 3 (MODERATE)' THEN 3
                WHEN CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 4 (SEVERE)' THEN 4
                WHEN CATEGORY LIKE '%STAGE 5%' THEN 5
                ELSE 0
            END AS DX_STAGE,
            NPI_NUMBER, COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID) AS PROVIDER_ID,
            PROVIDER_RENDERING_ID, PROVIDER_REFERRING_ID, PRIMARY_SPECIALTY_CODE
        FROM
        (
            SELECT C.NPI_NUMBER, A.PATIENT_ID, A.SERVICE_DATE, A.PROVIDER_RENDERING_ID, A.PROVIDER_REFERRING_ID, A.DIAGNOSIS_CODE,
                   B.PATIENT_BIRTH_YEAR, C.PRIMARY_SPECIALTY_CODE
            FROM
            (
                SELECT PATIENT_ID, CLAIM_ID, DIAGNOSIS_CODE, SERVICE_DATE, PROVIDER_RENDERING_ID, PROVIDER_REFERRING_ID
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX
            ) A
            LEFT JOIN
            (
                SELECT DISTINCT PATIENT_ID, PATIENT_BIRTH_YEAR, PATIENT_GENDER
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC
            ) B
            ON A.PATIENT_ID = B.PATIENT_ID
            LEFT JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR C
            ON COALESCE(A.PROVIDER_REFERRING_ID, A.PROVIDER_RENDERING_ID) = C.PROVIDER_ID
        ) DX
        INNER JOIN
        (
            SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION
            FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
            WHERE DIAGNOSIS_CODE IN ({DX03_CODES})
        ) CD
        ON DX.DIAGNOSIS_CODE = CD.DIAGNOSIS_CODE
        WHERE (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) >= {ANALYSIS_START_MONTH}
          AND (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) <= {END_MONTH}
          AND PROVIDER_ID IS NOT NULL
    """
    save_input(Get_Data_Snowflakes(query), "Dx_03")

### File Rx_01_new (Rx claims for adoption features)
Products limited to `CKD_PRODUCT_MAPPING_VW`, treatment category not 'other'. Rejected claims kept (`REJECTION_FLAG = 1`).

In [0]:
if PULL_RX_01_NEW:
    product_like = " OR ".join(f"RX.PRODUCT_NAME LIKE '%{p}%'" for p in ADOPTION_PRODUCTS)
    inv_far_filter = f"AND ({product_like})" if RX_ADOPTION_ONLY else ""
    query = f"""
        SELECT NPI_NUMBER, PROVIDER_ID, PRIMARY_SPECIALTY_CODE, RX.PATIENT_ID, SVC_DT, RX.PRODUCT_NAME, PRODUCT_STRENGTH,
            CASE WHEN TREATMENT_CATEGORY IN ('ACE INHIBITORS, PLAIN','ARBs') THEN 'ACE_or_ARB'
                 WHEN TREATMENT_CATEGORY IN ('CALCIUM CHANNEL BLOCKERS','BETA BLOCKING AGENTS') THEN 'Calcium_channel_or_Beta_blocker'
                 WHEN TREATMENT_CATEGORY IN ('ARBs + Diuretic','Diuretic + Blood Pressure','ACE inhibitors and diuretics','ACE inhibitors and calcium channel blockers',
                                             'ARBS+ Calcium Channel Blocker','ARBs + BETA BLOCKING AGENTS + diuretics','BETA BLOCKING AGENTS + Diuretic') THEN 'Combo_drug'
                 WHEN TREATMENT_CATEGORY IN ('Diuretic') THEN 'Diuretic_NON_MRA'
                 WHEN TREATMENT_CATEGORY IN ('Diuretic + MRA','SPIRONOLACTONE-MRA') THEN 'MRA'
                 ELSE TREATMENT_CATEGORY
            END AS TREATMENT_CATEGORY,
            MAX(DAYS_SUPPLY) AS DAYS_SUPPLY,
            CASE WHEN RX.PRODUCT_NAME LIKE '%INVOKANA%' AND PRODUCT_STRENGTH = '100 MG' THEN 'INVOKANA 100MG'
                 WHEN RX.PRODUCT_NAME LIKE '%INVOKANA%' AND PRODUCT_STRENGTH = '300 MG' THEN 'INVOKANA 300MG'
                 WHEN RX.PRODUCT_NAME LIKE '%JARDIANCE%' THEN 'JARDIANCE'
                 WHEN RX.PRODUCT_NAME LIKE '%FARXIGA%' THEN 'FARXIGA'
                 WHEN RX.PRODUCT_NAME LIKE '%SPIRONOLACTONE%' THEN 'SPIRONOLACTONE-MRA'
                 WHEN RX.PRODUCT_NAME IN ('ALOGLIPTIN BENZOATE', 'ALOGLIPTIN-METFORMIN HCL', 'ALOGLIPTIN-PIOGLITAZONE', 'SITAGLIPTIN-METFORMIN HCL', 'SITAGLIPTIN PHOSPHATE',
                      'LINAGLIPTIN-METFORMIN HCL', 'SAXAGLIPTIN-METFORMIN HCL', 'SAXAGLIPTIN HCL', 'LINAGLIPTIN', 'ALOGLIPTIN', 'ALOGLIPTIN/METFORMIN HCL', 'ALOGLIPTIN/PIOGLITAZONE',
                      'JANUMET', 'JANUMET XR', 'JANUVIA', 'JENTADUETO', 'JENTADUETO XR', 'KAZANO', 'KOMBIGLYZE XR', 'NESINA', 'ONGLYZA', 'OSENI', 'TRADJENTA') THEN 'DPP-IV'
                 WHEN RX.PRODUCT_NAME IN ('LIXISENATIDE', 'EXENATIDE', 'SEMAGLUTIDE', 'ALBIGLUTIDE', 'DULAGLUTIDE', 'LIRAGLUTIDE', 'INSULIN DEGLUDEC-LIRAGLUTIDE', 'ADLYXIN',
                      'ADLYXIN STARTER PACK', 'BYDUREON', 'BYDUREON BCISE', 'BYDUREON PEN', 'BYETTA', 'OZEMPIC', 'RYBELSUS', 'TANZEUM', 'TRULICITY', 'VICTOZA', 'XULTOPHY 100/3.6') THEN 'GLP-1'
                 ELSE Treatment_Category
            END AS Category,
            CASE WHEN REJECT_CODE IS NULL THEN 0 ELSE 1 END AS REJECTION_FLAG,
            METHOD_OF_PAYMENT, RX.PATIENT_BIRTH_YEAR, RX.PATIENT_GENDER,
            CASE WHEN REFILL_CODE = 0 THEN 'NEW_PRESCRIPTION' ELSE 'REFILL' END AS REFILL_FLAG
        FROM
        (
            SELECT DISTINCT C.NPI_NUMBER, A.PATIENT_ID, A.SVC_DT, E.METHOD_OF_PAYMENT, A.DAYS_SUPPLY, B.PATIENT_BIRTH_YEAR, B.PATIENT_GENDER, E.PAYER_NAME,
                   E.PAYER_PLAN_ID, E.PBM_NAME, E.PLAN_NAME, C.PRIMARY_SPECIALTY_CODE, C.PRIMARY_SPECIALTY_DESC, D.PRODUCT_NAME, D.PRODUCT_STRENGTH,
                   A.QUANTITY, A.REFILL_CODE, A.REJECT_CODE, A.PROVIDER_ID
            FROM
            (
                SELECT PATIENT_ID, PROVIDER_ID, SVC_DT, DAYS_SUPPLY, QUANTITY, REFILL_CODE, REJECT_CODE, NDC_CD, PAYER_PLAN_ID
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX
            ) A
            LEFT JOIN
            (
                SELECT DISTINCT PATIENT_ID, PATIENT_BIRTH_YEAR, PATIENT_GENDER
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC
            ) B
            ON A.PATIENT_ID = B.PATIENT_ID
            LEFT JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR C ON A.PROVIDER_ID = C.PROVIDER_ID
            LEFT JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD D ON A.NDC_CD = D.NDC_CD
            LEFT JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PLN E ON A.PAYER_PLAN_ID = E.PAYER_PLAN_ID
        ) RX
        JOIN PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW MAP
        ON RX.PRODUCT_NAME = MAP.PRODUCT_NAME
        WHERE (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) >= {ANALYSIS_START_MONTH}
          AND (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) <= {END_MONTH}
          AND PROVIDER_ID IS NOT NULL
          AND LOWER(Treatment_Category) NOT LIKE 'other'
          {inv_far_filter}
        GROUP BY NPI_NUMBER, PROVIDER_ID, PRIMARY_SPECIALTY_CODE, RX.PATIENT_ID, SVC_DT, RX.PRODUCT_NAME, PRODUCT_STRENGTH, Treatment_Category,
                 REJECTION_FLAG, METHOD_OF_PAYMENT, RX.PATIENT_BIRTH_YEAR, RX.PATIENT_GENDER, REFILL_FLAG
    """
    save_input(Get_Data_Snowflakes(query), "Rx_01_new")

In [0]:
import builtins   # 00_connection does `from pyspark.sql.functions import *`, which shadows round/min/max

checks = []
for name, flag in [("Dx_09", PULL_DX_09), ("Dx_03", PULL_DX_03), ("Rx_01_new", PULL_RX_01_NEW)]:
    if not flag:
        continue
    df = spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/{name}")
    s = df.agg(F.count("*").alias("rows"), F.countDistinct("PROVIDER_ID").alias("hcps"),
               F.min("SVC_DT").alias("min_dt"), F.max("SVC_DT").alias("max_dt"), F.max("PULL_DT").alias("pull_dt")).collect()[0]
    outside = df.select("PROVIDER_ID").distinct().join(hcp_universe, "PROVIDER_ID", "left_anti").count()
    checks.append({"file": name, "rows": s["rows"], "hcps": s["hcps"], "pct_universe": builtins.round(100 * s["hcps"] / n_hcp, 1),
                   "min_svc_dt": s["min_dt"], "max_svc_dt": s["max_dt"], "pull_dt": s["pull_dt"],
                   "hcps_outside_universe": outside,
                   "covers_last_cohort": s["max_dt"] >= lookback_end_max.date()})

checks_df = pd.DataFrame(checks)
display(checks_df)
assert (checks_df["rows"] > 0).all(), "Empty file"
assert (checks_df["hcps_outside_universe"] == 0).all(), "HCPs outside the 3.1 universe"
assert checks_df["covers_last_cohort"].all(), "Data does not reach the last cohort month"

if PULL_DX_09:
    display(spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/Dx_09").groupBy("DIAG_GRP")
            .agg(F.countDistinct("DIAGNOSIS_DESCRIPTION").alias("descriptions"), F.countDistinct("PATIENT_ID").alias("patients")))
if PULL_RX_01_NEW:
    rx_check = (spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/Rx_01_new")
                .groupBy("PRODUCT_NAME").agg(F.count("*").alias("rows"), F.countDistinct("PROVIDER_ID").alias("hcps"))
                .orderBy(F.desc("rows")))
    display(rx_check)
    found = {r["PRODUCT_NAME"] for r in rx_check.collect()}
    missing = [p for p in ADOPTION_PRODUCTS if p not in found]
    assert not missing, f"Adoption products missing from Rx_01_new: {missing}"